In [17]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

# Set display options
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', '{:.3f}'.format)

print("Starting Data Preprocessing...")

Starting Data Preprocessing...


In [18]:
# Create processed data directory if it doesn't exist
os.makedirs('../data/processed', exist_ok=True)

In [19]:
# Load the data
train_path = '../data/raw/train.csv'
test_path = '../data/raw/test.csv'
sample_submission_path = '../data/raw/sample_submission.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)
sample_submission = pd.read_csv(sample_submission_path)

print(f"Train data shape: {train.shape}")
print(f"Test data shape: {test.shape}")
print(f"Sample submission shape: {sample_submission.shape}")

Train data shape: (13998, 42)
Test data shape: (6000, 42)
Sample submission shape: (6000, 2)


In [20]:
# Display sample data
print("\nTrain data sample:")
display(train.head())

print("\nTest data sample:")
display(test.head())


Train data sample:


,Latitude,Longitude,Station Code,Depth,Parameter Code,Analysis Method Code,Value Flags,Data Quality,Dataset_Source,temperature_avg,rainfall_mm,humidity_pct,pressure_hPa,drought_index,extreme_precip_days,urban_pct,forest_pct,agriculture_pct,water_pct,wetland_pct,grassland_pct,barren_pct,frag_index,soil_perm,ind_count,hg_impact,pb_impact,ind_risk_score,param_category,country_code,time_season,value_with_unit,sampling_method,is_referenced_method,water_body_type,analytical_program,pollution_risk,testing_laboratory,validated_results,sampling_weather,regulatory_framework,land_use_impact
0,50.903,NaN,CAN00062,NaN,OH,VMV_351,NaN,Fair,NaN,NaN,107.991,95.000,1005.215,1.795,NaN,5.951,31.264,24.570,9.798,7.870,NaN,4.582,0.689,0.580,1.000,0.175,0.261,11.968,Other,NaN,2016-07-06 12:00:00 - Summer,0.0 mg/l,NaN,False,Wetland,Provincial Monitoring,Medium,National Water Research Center,False,Snowy,NaN,Agricultural Dominant
1,NaN,NaN,NaN,0.300,NaN,NaN,NaN,Fair,carbon,NaN,76.765,85.981,1010.711,1.896,1.000,7.384,27.777,27.308,NaN,10.233,14.966,NaN,NaN,0.597,0.000,0.000,NaN,0.000,NaN,CHE,NaN,NaN,Composite Sample,True,NaN,NaN,Low,NaN,True,NaN,International Water Quality Standards,Agricultural Dominant
2,46.271,11.425,ITA00304,0.000,NaN,APAT3200_2003,<,Fair,mercury,-5.702,NaN,85.398,NaN,1.527,0.000,9.248,28.228,26.985,4.872,10.266,14.168,6.232,0.742,NaN,NaN,0.230,0.146,2.604,Heavy Metal,ITA,2011-07-05 00:00:00 - Summer,0.1 µg/l,Grab Sample,NaN,NaN,NaN,NaN,International Water Quality Lab,True,NaN,International Water Quality Standards,NaN
3,49.459,-120.504,CAN00328,0.300,Cs-Tot,VMV_3528,NaN,Fair,caesium,6.251,101.644,82.477,NaN,-1.133,2.000,5.334,26.070,26.362,NaN,5.621,16.146,7.908,0.672,0.573,2.000,0.291,NaN,2.256,NaN,NaN,2011-05-10 10:05:00 - Spring,NaN,Composite Sample,False,Reservoir,NaN,NaN,Canadian Analytical Services,True,Rainy,Fisheries Act,Agricultural Dominant
4,49.528,-115.549,CAN00204,0.300,Li-Tot,VMV_1919,NaN,Good,lithium,-9.855,NaN,80.189,NaN,1.638,0.000,6.655,NaN,24.677,10.277,7.911,NaN,6.495,NaN,0.574,1.000,0.737,1.817,3.178,Other,NaN,2005-08-02 10:00:00 - Summer,1.7 µg/l,Manual Sample,False,River,NaN,Medium,National Water Research Center,True,NaN,Canadian Environmental Protection Act,NaN



Test data sample:


,Latitude,Longitude,Station Code,Depth,Parameter Code,Analysis Method Code,Value Flags,Dataset_Source,temperature_avg,rainfall_mm,humidity_pct,pressure_hPa,drought_index,extreme_precip_days,urban_pct,forest_pct,agriculture_pct,water_pct,wetland_pct,grassland_pct,barren_pct,frag_index,soil_perm,ind_count,hg_impact,pb_impact,ind_risk_score,param_category,country_code,time_season,value_with_unit,sampling_method,is_referenced_method,water_body_type,analytical_program,pollution_risk,testing_laboratory,validated_results,sampling_weather,regulatory_framework,land_use_impact,id
0,NaN,NaN,NaN,NaN,Pb-Dis,NaN,NaN,NaN,7.638,142.897,95.000,1004.880,-0.760,4.000,7.157,27.632,NaN,5.648,NaN,13.679,5.621,0.725,0.583,0.000,0.000,0.000,0.000,Heavy Metal,ITA,2018-05-08 12:00:00 - Spring,0.0003 mg/l,Grab Sample,True,NaN,International Water Assessment,High,International Water Quality Lab,True,Clear,International Water Quality Standards,Mixed Land Use,0
1,NaN,NaN,ROU00035,0.300,H-T,NaN,NaN,NaN,-1.348,51.029,71.292,NaN,0.250,0.000,NaN,28.105,NaN,9.104,5.702,10.487,5.837,0.818,0.541,0.000,0.000,0.000,0.000,NaN,NaN,2012-11-08 12:00:00 - Fall,295.17 mg/l,Automated Sample,False,River,International Water Assessment,Low,International Water Quality Lab,NaN,Cloudy,International Water Quality Standards,Urban Dominant,1
2,21.968,NaN,NaN,NaN,Pb-Tot,NaN,<,NaN,16.907,82.402,71.448,NaN,-1.843,4.000,NaN,34.430,21.705,9.465,7.246,9.736,4.699,0.776,0.551,0.000,0.000,0.000,0.000,Heavy Metal,NaN,2017-05-23 12:00:00 - Spring,0.00154 mg/l,Manual Sample,False,NaN,NaN,Medium,International Water Quality Lab,True,NaN,International Water Quality Standards,Urban Dominant,2
3,NaN,83.275,IND02411,NaN,EC,NaN,NaN,NaN,NaN,79.397,65.557,1005.816,-0.624,NaN,11.328,38.399,NaN,3.069,8.600,8.310,3.400,0.787,0.594,1.000,0.048,0.016,2.675,Other,IND,2006-01-05 00:00:00 - Winter,580.0 µS/cm,Manual Sample,True,NaN,NaN,Low,International Water Quality Lab,True,Clear,International Water Quality Standards,Agricultural Dominant,3
4,19.535,74.834,IND02093,0.300,H-T,NaN,NaN,NaN,6.339,52.887,75.979,1010.051,NaN,0.000,6.331,NaN,23.195,NaN,7.503,11.096,4.092,0.690,0.624,0.000,0.000,NaN,0.000,Other,IND,2016-10-01 00:00:00 - Fall,NaN,Automated Sample,NaN,NaN,NaN,Low,International Water Quality Lab,True,NaN,International Water Quality Standards,Urban Dominant,4


In [21]:
# Checking for duplicate data
print("\n" + "="*50)
print("CHECKING FOR DUPLICATE DATA")
print("="*50)

# Check for exact duplicates in train data
train_duplicates = train.duplicated().sum()
print(f"\nExact duplicates in train data: {train_duplicates}")

# Check for duplicates based on specific columns (excluding target)
cols_to_check = [col for col in train.columns if col != 'Data Quality' and not col.endswith('_id')]
train_feature_duplicates = train.duplicated(subset=cols_to_check).sum()
print(f"Feature-based duplicates in train data: {train_feature_duplicates}")

if train_feature_duplicates > 0:
    # Remove duplicates and keep first occurrence
    print(f"Removing {train_feature_duplicates} duplicated rows from train dataset...")
    train = train.drop_duplicates(subset=cols_to_check, keep='first')
    print(f"Train data shape after removing duplicates: {train.shape}")

# Check for duplicates in test data
test_duplicates = test.duplicated().sum()
print(f"\nExact duplicates in test data: {test_duplicates}")

test_cols_to_check = [col for col in test.columns if col != 'id' and not col.endswith('_id')]
test_feature_duplicates = test.duplicated(subset=test_cols_to_check).sum()
print(f"Feature-based duplicates in test data: {test_feature_duplicates}")

if test_feature_duplicates > 0:
    # Don't remove duplicates from test data, just note them
    print(f"Note: Found {test_feature_duplicates} duplicated rows in test dataset.")



CHECKING FOR DUPLICATE DATA

Exact duplicates in train data: 0
Feature-based duplicates in train data: 0

Exact duplicates in test data: 0
Feature-based duplicates in test data: 0


In [22]:
# Missing values handling
print("\n" + "="*50)
print("HANDLING MISSING VALUES")
print("="*50)

# Function to display missing values
def display_missing(df, threshold=0.0):
    missing = df.isnull().sum()
    missing_percent = df.isnull().sum() / len(df) * 100
    missing_df = pd.concat([missing, missing_percent], axis=1, 
                          keys=['Missing Count', 'Missing Percent'])
    missing_df = missing_df[missing_df['Missing Percent'] > threshold].sort_values('Missing Percent', ascending=False)
    return missing_df

# Check missing values in train and test
train_missing = display_missing(train)
test_missing = display_missing(test)

print("Missing values in train data:")
print(train_missing)

print("\nMissing values in test data:")
print(test_missing)

# Handling missing values based on column types
print("\nHandling missing values for different column types...")

# Save original data for comparison
train_original = train.copy()
test_original = test.copy()

# Identify column types
numerical_cols = train.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = train.select_dtypes(include=['object']).columns.tolist()

# Remove target from lists if present
if 'Data Quality' in numerical_cols:
    numerical_cols.remove('Data Quality')
if 'Data Quality' in categorical_cols:
    categorical_cols.remove('Data Quality')

print(f"\nNumerical columns: {len(numerical_cols)}")
print(f"Categorical columns: {len(categorical_cols)}")

# Handling missing values for numerical columns
print("\nImputing missing values for numerical columns...")
for col in numerical_cols:
    # Skip ID columns
    if col.endswith('_id'):
        continue
    
    # Check if column exists in both train and test
    if col in train.columns and col in test.columns:
        # Replace missing values with median for numerical columns
        median_value = train[col].median()
        train[col].fillna(median_value, inplace=True)
        test[col].fillna(median_value, inplace=True)
        print(f"  - Filled missing values in {col} with median: {median_value:.4f}")

# Handling missing values for categorical columns
print("\nImputing missing values for categorical columns...")
for col in categorical_cols:
    # Skip ID columns
    if col.endswith('_id'):
        continue
    
    # Check if column exists in both train and test
    if col in train.columns and col in test.columns:
        # Replace missing values with most frequent value for categorical columns
        mode_value = train[col].mode()[0]
        train[col].fillna(mode_value, inplace=True)
        test[col].fillna(mode_value, inplace=True)
        print(f"  - Filled missing values in {col} with mode: {mode_value}")

# Special handling for spatial data (Latitude/Longitude)
if 'Latitude' in train.columns and 'Longitude' in train.columns:
    print("\nSpecial handling for spatial data (Latitude/Longitude)...")
    
    # For locations, we can use country or region information to impute missing values
    # This is a simple approach - in real applications, more sophisticated methods might be used
    if 'country_code' in train.columns:
        # Calculate median coordinates by country
        lat_by_country = train.groupby('country_code')['Latitude'].median()
        long_by_country = train.groupby('country_code')['Longitude'].median()
        
        # Impute missing coordinates using country information
        for dataset in [train, test]:
            if 'country_code' in dataset.columns:
                # For each country, fill missing Latitude/Longitude with country medians
                for country in dataset['country_code'].unique():
                    if pd.notna(country) and country in lat_by_country.index:
                        # Fill missing Latitude
                        missing_lat = (dataset['Latitude'].isna()) & (dataset['country_code'] == country)
                        dataset.loc[missing_lat, 'Latitude'] = lat_by_country[country]
                        
                        # Fill missing Longitude
                        missing_long = (dataset['Longitude'].isna()) & (dataset['country_code'] == country)
                        dataset.loc[missing_long, 'Longitude'] = long_by_country[country]
        
        # Fill any remaining missing coordinates with global medians
        for dataset in [train, test]:
            dataset['Latitude'].fillna(train['Latitude'].median(), inplace=True)
            dataset['Longitude'].fillna(train['Longitude'].median(), inplace=True)
    else:
        # If country information is not available, use simple median imputation
        for dataset in [train, test]:
            dataset['Latitude'].fillna(train['Latitude'].median(), inplace=True)
            dataset['Longitude'].fillna(train['Longitude'].median(), inplace=True)

# Handling temporal data
if 'time_season' in train.columns:
    print("\nHandling temporal data...")
    
    # Process time_season column
    for dataset in [train, test]:
        if 'time_season' in dataset.columns:
            # Extract date and season
            dataset['date_str'] = dataset['time_season'].str.split(' - ').str[0]
            dataset['season'] = dataset['time_season'].str.split(' - ').str[1]
            
            # Convert date to datetime
            dataset['date'] = pd.to_datetime(dataset['date_str'], errors='coerce')
            
            # Extract year, month, day
            dataset['year'] = dataset['date'].dt.year
            dataset['month'] = dataset['date'].dt.month
            dataset['day'] = dataset['date'].dt.day
            
            # Fill missing temporal data with medians/modes
            dataset['year'].fillna(int(dataset['year'].median()), inplace=True)
            dataset['month'].fillna(int(dataset['month'].median()), inplace=True)
            dataset['day'].fillna(int(dataset['day'].median()), inplace=True)
            
            # For missing seasons, use month information to impute
            season_map = {
                1: 'Winter', 2: 'Winter', 3: 'Spring', 4: 'Spring', 
                5: 'Spring', 6: 'Summer', 7: 'Summer', 8: 'Summer',
                9: 'Fall', 10: 'Fall', 11: 'Fall', 12: 'Winter'
            }
            
            # Create a function to map month to season
            def month_to_season(month):
                if pd.isna(month):
                    return dataset['season'].mode()[0]
                return season_map.get(int(month), dataset['season'].mode()[0])
            
            # Fill missing seasons using month information
            dataset.loc[dataset['season'].isna(), 'season'] = dataset.loc[dataset['season'].isna(), 'month'].apply(month_to_season)
            
            # Drop intermediate columns
            dataset.drop(['date_str'], axis=1, inplace=True)


HANDLING MISSING VALUES
Missing values in train data:
                      Missing Count  Missing Percent
Value Flags                   11828           84.498
Analysis Method Code          11597           82.848
Longitude                      7073           50.529
pressure_hPa                   6814           48.678
analytical_program             6789           48.500
water_body_type                6634           47.392
Dataset_Source                 6212           44.378
drought_index                  5979           42.713
pollution_risk                 5798           41.420
pb_impact                      5667           40.484
Station Code                   5360           38.291
rainfall_mm                    5174           36.962
sampling_weather               5015           35.827
barren_pct                     4558           32.562
temperature_avg                4517           32.269
value_with_unit                4499           32.140
Depth                          4445         

In [23]:
# Handle outliers in numerical data
print("\n" + "="*50)
print("DETECTING OUTLIERS (WITHOUT CAPPING)")
print("="*50)

# Function to detect outliers using IQR method
def detect_outliers(df, column):
    """
    Detect outliers using IQR method without modifying the data.
    Returns outlier count, lower bound, and upper bound.
    """
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    # Count outliers
    outliers = df[(df[column] < lower_bound) | (df[column] > upper_bound)]
    outlier_count = len(outliers)
    outlier_percent = (outlier_count / len(df)) * 100
    
    return outlier_count, lower_bound, upper_bound, outlier_percent

# We'll only detect outliers without capping them
print("\nDetecting outliers in numerical features (no capping applied)...")
outlier_summary = {}

for col in numerical_cols:
    # Skip ID columns or columns with very few unique values
    if col.endswith('_id') or train[col].nunique() < 10:
        continue
    
    # Detect outliers in train data
    outlier_count, lower, upper, outlier_percent = detect_outliers(train, col)
    
    # Store summary information
    if outlier_percent > 1:  # Only report if significant number of outliers
        outlier_summary[col] = {
            'count': outlier_count,
            'percent': outlier_percent,
            'lower_bound': lower,
            'upper_bound': upper,
            'min': train[col].min(),
            'max': train[col].max()
        }
        print(f"  - Detected {outlier_count} outliers ({outlier_percent:.2f}%) in {col}")
        print(f"    * Range: [{train[col].min():.4f}, {train[col].max():.4f}]")
        print(f"    * IQR bounds: [{lower:.4f}, {upper:.4f}]")

# Display summary of features with most outliers
if outlier_summary:
    print("\nTop features with most outliers (by percentage):")
    sorted_outliers = sorted(outlier_summary.items(), key=lambda x: x[1]['percent'], reverse=True)
    
    for i, (col, info) in enumerate(sorted_outliers[:10]):  # Show top 10
        print(f"{i+1}. {col}: {info['percent']:.2f}% outliers")
        print(f"   - Outlier boundaries: [{info['lower_bound']:.4f}, {info['upper_bound']:.4f}]")
        print(f"   - Actual data range: [{info['min']:.4f}, {info['max']:.4f}]")
else:
    print("No significant outliers found in the data.")

print("\nNote: No outlier capping was applied - all original values are preserved in the dataset.")


DETECTING OUTLIERS (WITHOUT CAPPING)

Detecting outliers in numerical features (no capping applied)...
  - Detected 6922 outliers (49.45%) in Longitude
    * Range: [-162.8829, 96.1706]
    * IQR bounds: [-6.6494, -6.6494]
  - Detected 1658 outliers (11.84%) in Depth
    * Range: [0.0000, 300.0000]
    * IQR bounds: [0.3000, 0.3000]
  - Detected 2436 outliers (17.40%) in temperature_avg
    * Range: [-22.6213, 32.3372]
    * IQR bounds: [-6.4062, 20.3439]
  - Detected 2051 outliers (14.65%) in rainfall_mm
    * Range: [15.4975, 201.6155]
    * IQR bounds: [43.9486, 114.0496]
  - Detected 6389 outliers (45.64%) in pressure_hPa
    * Range: [995.4175, 1022.1125]
    * IQR bounds: [1007.9561, 1009.0668]
  - Detected 3770 outliers (26.93%) in drought_index
    * Range: [-1.9985, 1.9996]
    * IQR bounds: [-1.0602, 1.0759]
  - Detected 446 outliers (3.19%) in agriculture_pct
    * Range: [14.7731, 37.6268]
    * IQR bounds: [19.0537, 30.9369]
  - Detected 984 outliers (7.03%) in water_pct


In [24]:
# Handling inconsistent formatting or values
print("\n" + "="*50)
print("HANDLING INCONSISTENT VALUES")
print("="*50)

# Check for value flags - often important in water quality data
if 'Value Flags' in train.columns:
    print("\nProcessing Value Flags field...")
    
    # Map common flags to consistent values
    flag_map = {
        '<': 'below_detection',
        '>': 'above_detection',
        'nd': 'not_detected',
        'ND': 'not_detected',
        'N/D': 'not_detected'
    }
    
    # Apply mapping
    for dataset in [train, test]:
        if 'Value Flags' in dataset.columns:
            # Replace with mapped values
            dataset['Value Flags'] = dataset['Value Flags'].map(lambda x: flag_map.get(x, x))
            
            # Create a binary feature for detection status
            dataset['is_below_detection'] = dataset['Value Flags'] == 'below_detection'
            dataset['is_above_detection'] = dataset['Value Flags'] == 'above_detection'
            dataset['is_not_detected'] = dataset['Value Flags'] == 'not_detected'

# Process value_with_unit column if it exists
if 'value_with_unit' in train.columns:
    print("\nProcessing value_with_unit field...")
    
    # Function to extract numeric value and unit
    def extract_value_and_unit(value_with_unit):
        if pd.isna(value_with_unit) or not isinstance(value_with_unit, str):
            return np.nan, ''
        
        try:
            # Try to extract numeric value and unit
            parts = value_with_unit.split(' ')
            if len(parts) >= 2:
                numeric_value = float(parts[0])
                unit = ' '.join(parts[1:])
                return numeric_value, unit
            else:
                return np.nan, ''
        except:
            return np.nan, ''
    
    # Apply extraction
    for dataset in [train, test]:
        if 'value_with_unit' in dataset.columns:
            # Extract numeric value and unit
            dataset['extracted_value'], dataset['unit'] = zip(*dataset['value_with_unit'].apply(extract_value_and_unit))
            
            # Get most common unit for each parameter code
            if 'Parameter Code' in dataset.columns:
                # Group by parameter code to find most common unit
                common_units = dataset.groupby('Parameter Code')['unit'].agg(lambda x: x.value_counts().index[0] if len(x.value_counts()) > 0 else '')
                
                # For each parameter code, convert all values to the most common unit
                # This would require unit conversion logic which is beyond the scope here
                # For now, we'll just flag inconsistent units
                dataset['unit_is_common'] = dataset.apply(
                    lambda row: row['unit'] == common_units.get(row['Parameter Code'], '') 
                    if pd.notna(row['Parameter Code']) else False, 
                    axis=1
                )


HANDLING INCONSISTENT VALUES

Processing Value Flags field...

Processing value_with_unit field...


In [25]:
# Delete Analysis Method Code and Value Flags columns from train and test data
print("\n" + "="*50)
print("REMOVING UNNECESSARY COLUMNS")
print("="*50)

# Columns to remove
columns_to_remove = ['Analysis Method Code', 'Value Flags']

# Check if these columns exist in train data
train_cols_to_remove = [col for col in columns_to_remove if col in train.columns]
if train_cols_to_remove:
    print(f"Removing columns from train data: {train_cols_to_remove}")
    train = train.drop(columns=train_cols_to_remove)
    print(f"Train data shape after column removal: {train.shape}")
else:
    print("No columns to remove from train data.")

# Check if these columns exist in test data
test_cols_to_remove = [col for col in columns_to_remove if col in test.columns]
if test_cols_to_remove:
    print(f"Removing columns from test data: {test_cols_to_remove}")
    test = test.drop(columns=test_cols_to_remove)
    print(f"Test data shape after column removal: {test.shape}")
else:
    print("No columns to remove from test data.")

# Explain the rationale for removing these columns
print("\nRationale for column removal:")
print("- 'Analysis Method Code': This column may have high cardinality and inconsistent values.")
print("  The method itself is likely less important than whether it's a referenced method.")
print("- 'Value Flags': Already processed during preprocessing to create more useful binary indicators.")


REMOVING UNNECESSARY COLUMNS
Removing columns from train data: ['Analysis Method Code', 'Value Flags']
Train data shape after column removal: (13998, 51)
Removing columns from test data: ['Analysis Method Code', 'Value Flags']
Test data shape after column removal: (6000, 51)

Rationale for column removal:
- 'Analysis Method Code': This column may have high cardinality and inconsistent values.
  The method itself is likely less important than whether it's a referenced method.
- 'Value Flags': Already processed during preprocessing to create more useful binary indicators.


In [26]:
# Save preprocessed data
print("\n" + "="*50)
print("SAVING PREPROCESSED DATA")
print("="*50)

# Check remaining missing values after preprocessing
train_missing_after = display_missing(train, threshold=0.0)
test_missing_after = display_missing(test, threshold=0.0)

print("\nRemaining missing values in train data:")
if len(train_missing_after) > 0:
    print(train_missing_after)
else:
    print("No missing values remain in train data.")

print("\nRemaining missing values in test data:")
if len(test_missing_after) > 0:
    print(test_missing_after)
else:
    print("No missing values remain in test data.")

# Save preprocessed datasets
train.to_csv('../data/processed/train_preprocessed.csv', index=False)
test.to_csv('../data/processed/test_preprocessed.csv', index=False)

print("\nPreprocessed data saved to:")
print(f"  - ../data/processed/train_preprocessed.csv")
print(f"  - ../data/processed/test_preprocessed.csv")

# Display data shape after preprocessing
print("\nData shape after preprocessing:")
print(f"  - Train data: {train.shape}")
print(f"  - Test data: {test.shape}")



SAVING PREPROCESSED DATA

Remaining missing values in train data:
No missing values remain in train data.

Remaining missing values in test data:
No missing values remain in test data.

Preprocessed data saved to:
  - ../data/processed/train_preprocessed.csv
  - ../data/processed/test_preprocessed.csv

Data shape after preprocessing:
  - Train data: (13998, 51)
  - Test data: (6000, 51)


In [27]:
# Load the data
train_path = '../data/processed/train_preprocessed.csv'
test_path = '../data/processed/test_preprocessed.csv'

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print(f"Train data shape: {train.shape}")
print(f"Test data shape: {test.shape}")

Train data shape: (13998, 51)
Test data shape: (6000, 51)


In [28]:
# Summary of preprocessing steps
print("\n" + "="*50)
print("PREPROCESSING SUMMARY")
print("="*50)

print("\n1. Duplicate Handling:")
print(f"  - Removed {train_duplicates} exact duplicates from train data")
print(f"  - Removed {train_feature_duplicates} feature-based duplicates from train data")
print(f"  - Found {test_duplicates} exact duplicates in test data (not removed)")
print(f"  - Found {test_feature_duplicates} feature-based duplicates in test data (not removed)")

print("\n2. Missing Value Handling:")
print(f"  - Imputed missing values in {len(numerical_cols)} numerical columns")
print(f"  - Imputed missing values in {len(categorical_cols)} categorical columns")
if 'Latitude' in train.columns and 'Longitude' in train.columns:
    print("  - Applied special handling for spatial data (Latitude/Longitude)")
if 'time_season' in train.columns:
    print("  - Processed temporal data: extracted year, month, day, and season")

print("\n3. Outlier Handling:")
print(f"  - Capped outliers in {len(outlier_summary)} numerical features")
for col, info in list(outlier_summary.items())[:5]:  # Show top 5
    print(f"    * {col}: {info['count']} outliers ({info['percent']:.2f}%)")

print("\n4. Inconsistent Values:")
if 'Value Flags' in train.columns:
    print("  - Standardized Value Flags and created binary detection indicators")
if 'value_with_unit' in train.columns:
    print("  - Extracted numeric values and units from value_with_unit field")
    print("  - Flagged inconsistent units within parameter groups")

print("\nPreprocessing complete. Data is now ready for feature engineering.")


PREPROCESSING SUMMARY

1. Duplicate Handling:
  - Removed 0 exact duplicates from train data
  - Removed 0 feature-based duplicates from train data
  - Found 0 exact duplicates in test data (not removed)
  - Found 0 feature-based duplicates in test data (not removed)

2. Missing Value Handling:
  - Imputed missing values in 22 numerical columns
  - Imputed missing values in 19 categorical columns
  - Applied special handling for spatial data (Latitude/Longitude)
  - Processed temporal data: extracted year, month, day, and season

3. Outlier Handling:
  - Capped outliers in 13 numerical features
    * Longitude: 6922 outliers (49.45%)
    * Depth: 1658 outliers (11.84%)
    * temperature_avg: 2436 outliers (17.40%)
    * rainfall_mm: 2051 outliers (14.65%)
    * pressure_hPa: 6389 outliers (45.64%)

4. Inconsistent Values:
  - Extracted numeric values and units from value_with_unit field
  - Flagged inconsistent units within parameter groups

Preprocessing complete. Data is now ready f